# Tables 2, 8, 9 and 10

The word directions of the 200 class names against the visual directions of the classes they name.

Inputs: `scripts/analysis/table2_class_names.py`, which writes `results/class_alignment.parquet`, `results/class_retrieval.parquet`, `results/class_checks.parquet`, `results/class_diagnostics.parquet` and `results/class_depth.parquet`; `results/purity.parquet` and `results/retrieval.parquet` from `scripts/analysis/table1_models.py`, for the Table 1 values Section 5 compares; the committed candidate list `lexicon/class_names.tsv`; `data/lexicon/concepts.parquet` from `scripts/stimuli/class_names.py`; `data/concepts/images.parquet` from `scripts/stimuli/class_images.py`; `data/lexicon/words.parquet` from `scripts/stimuli/vocabulary.py`; the ImageNet-1k class names.

In [ ]:
import numpy as np
import pandas as pd

from src.analysis.directions import CLASS_IMAGES, manifest
from src.config import CLASS_NAMES_TSV
from src.models import PARAMS_M, PRETTY
from src.paper import tables
from src.paper.results import class_names, read, sweep_rows, words
from src.paper.style import setup_notebook
from src.stats import binomial_se
from src.stimuli import imagenet
from src.stimuli.typography import n_slots

setup_notebook()

alignment = read("class_alignment")
retrieval = read("class_retrieval")
checks = read("class_checks")
classes = read("class_diagnostics")
depth = read("class_depth")
purity = read("purity")
forward = read("retrieval").query("direction == 'forward'")

final = alignment[alignment.subset.eq("all")].set_index("model")
text_free = alignment[alignment.subset.eq("text_free")].set_index("model")

## Table 2

Matched cosine similarity and retrieval in both directions at the final layer, on all class images.

In [ ]:
table2 = tables.table2_class_names(alignment, retrieval)
table2.write()
table2.frame

In [ ]:
ranked = retrieval[retrieval.subset.eq("all")].set_index(["direction", "model"])
word_to_class = ranked.loc["word_to_class"]
class_to_word = ranked.loc["class_to_word"]
print(f"class names: {word_to_class.n.iloc[0]}")
print(f"visual directions ranked for each word direction: {word_to_class.n_candidates.iloc[0]:,}")
print(f"word directions ranked for each named class: {class_to_word.n_candidates.iloc[0]}")
for model in ("dinov3-h", "dinov3-7b", "clip-b", "siglip2-b", "dinov3-l"):
    print(f"matched cosine similarity, {PRETTY[model]}: {final.alignment[model]:.3f}")
h = word_to_class.loc["dinov3-h"]
print(f"DINOv3-H word to class: median rank {h.median_rank:g}, "
      f"named class ranked first for {h.hits} of {h.n} ({h.hits / h.n:.2f})")
ratios = pd.Series({
    "matched cosine similarity": final.alignment["dinov3-h"] / final.alignment["dinov3-7b"],
    "word-to-class top-10 accuracy": word_to_class.rate10["dinov3-h"] / word_to_class.rate10["dinov3-7b"],
    "class-to-word top-10 accuracy": class_to_word.rate10["dinov3-h"] / class_to_word.rate10["dinov3-7b"]})
for name, ratio in ratios.items():
    print(f"DINOv3-H over DINOv3-7B, {name}: {ratio:.2f}")
print(f"DINOv3-H over DINOv3-7B: {ratios.min():.1f} to {ratios.max():.1f} times")


def table1_value(model, relations):
    """A Table 1 value and its standard error, as tables.table1_models computes them."""
    if relations[0] in ("category", "brand"):
        row = purity[purity.model.eq(model) & purity.axis.eq(relations[0])].iloc[0]
        return float(row.purity), float(row.se)
    rows = forward[forward.model.eq(model) & forward.relation.isin(relations)]
    hits, n = int(rows.hits10.sum()), int(rows.n.sum())
    return hits / n, binomial_se(hits, n)


TABLE1_NAMES = ("nouns", "brands", "capital", "synonym", "one-edit word", "plural or base")
pairs = {name: (table1_value("dinov3-7b", relations), table1_value("dinov3-h", relations))
         for name, relations in zip(TABLE1_NAMES, tables.TABLE1_COLUMNS)}
print("Table 1, DINOv3-7B and DINOv3-H: "
      + ", ".join(f"{name} {a:.3f} and {b:.3f}" for name, ((a, _), (b, _)) in pairs.items()))
for name, ((a, sa), (b, sb)) in pairs.items():
    if a < b:
        print(f"DINOv3-7B below DINOv3-H in Table 1: {name}, by {(b - a) / np.hypot(sa, sb):.2f} "
              f"standard errors of the difference")

## Table 8

Matched cosine similarity with its permutation null, on all class images and on class images with no detected text.

In [ ]:
table8 = tables.table8_class_alignment(alignment)
table8.write()
table8.frame

In [ ]:
n_permutations = int(final.n_permutations.iloc[0])
aligned = final.p.lt(0.05)
print(f"random reassignments of classes to words: {n_permutations:,}")
print(f"largest p of {', '.join(PRETTY[model] for model in final.index[aligned])}: "
      f"{final.p[aligned].max():.4f}")
print(f"smallest p of the other {int((~aligned).sum())} models: {final.p[~aligned].min():.3f}")
one_edit = forward[forward.relation.eq("spelling_twin")].set_index("model")
rest = one_edit.rate10[final.index[~aligned]]
print(f"one-edit word top-10 accuracy of the other {len(rest)} models: at least {rest.min():.3f} "
      f"({PRETTY[rest.idxmin()]}), chance {one_edit.chance10.iloc[0]:.3f}")
for model in ("webssl-dino-1b", "dinov2-g"):
    print(f"{PRETTY[model]}: {PARAMS_M[model]:,} million parameters, p = {final.p[model]:.3f}")
print(f"class names whose classes keep at least {imagenet.MIN_IMAGES} images with no detected text: "
      f"{text_free.n_words.iloc[0]}")
print(f"matched cosine similarity with no detected text, DINOv3-H: {text_free.alignment['dinov3-h']:.3f}")

## Table 9

Top-10 accuracy and median rank in both directions, and top-10 accuracy on class images with no detected text.

In [ ]:
table9 = tables.table9_class_retrieval(retrieval)
table9.write()
table9.frame

In [ ]:
subsets = {"all": "all class images", "text_free": "no detected text"}
for (subset, direction), rows in retrieval.groupby(["subset", "direction"], sort=False):
    print(f"{subsets[subset]}, {direction.replace('_', ' ')}: {rows.n.iloc[0]} queries, "
          f"{rows.n_candidates.iloc[0]:,} candidates, top-10 chance {rows.chance10.iloc[0]:.3f}")

## Table 10

Matched cosine similarity and word-to-class top-10 accuracy at the nine depths after the input.

In [ ]:
table10 = tables.table10_class_depth(depth)
table10.write()
table10.frame

In [ ]:
relative = depth.groupby("layer_index").relative_depth
label = relative.mean()
print(f"depths after the input: {len(label)}, labeled {', '.join(f'{d:.2f}' for d in label)}")
print(f"widest range of relative depth within a column: {(relative.max() - relative.min()).max():.3f}")
last = depth.layer_index.max()
layer = depth.pivot(index="model", columns="layer_index", values="layer")
n_layers = depth.groupby("model").n_layers.first()
for n, gap in (layer[last] - layer[last - 1]).groupby(n_layers).unique().items():
    print(f"layers between the last two depths in the {n}-layer models: {', '.join(map(str, gap))}")
print(f"word-to-class top-10 chance: {depth.word_to_class_chance10.iloc[0]:.3f}")

early = depth[depth.layer_index.le(last - 2)]
peak = early.loc[early.alignment.abs().idxmax()]
print(f"largest absolute matched cosine similarity through relative depth {label[last - 2]:.2f}: "
      f"{abs(peak.alignment):.3f}, in {PRETTY[peak.model]} at {label[peak.layer_index]:.2f}")
null = final.index[~aligned]
print(f"largest absolute matched cosine similarity at any depth in the {len(null)} models with p above 0.05: "
      f"{depth[depth.model.isin(null)].alignment.abs().max():.3f}")

before = depth[depth.layer_index.lt(last)].groupby("model").word_to_class_rate10.max()
others = before.drop("siglip2-b")
print(f"largest word-to-class top-10 accuracy before the last depth, apart from SigLIP2-B: "
      f"{others.max():.3f}, in {PRETTY[others.idxmax()]}")
siglip = depth[depth.model.eq("siglip2-b")].set_index("layer_index").word_to_class_rate10
print(f"SigLIP2-B word-to-class top-10 accuracy: {siglip[last - 1]:.3f} at the next-to-last depth, "
      f"{siglip[last]:.3f} at the last")

## The class names and their images

The candidate names, the names that satisfy the selection rules, the kept names and the class images.

In [ ]:
candidates = pd.read_csv(CLASS_NAMES_TSV, sep="\t")
kept = candidates.decision.eq("keep")
below_text_bar = candidates.reason.str.startswith("below the text bar")
print(f"candidate names: {len(candidates)}")
print(f"names that satisfy the selection rules: {int((kept | below_text_bar).sum())}")
print(f"class names kept: {int(kept.sum())}")
print(f"share of class images with no detected text: at least {candidates.text_free[kept].min():.2f} "
      f"for the {int(kept.sum())} kept, at most {candidates.text_free[below_text_bar].max():.2f} "
      f"for the other {int(below_text_bar.sum())}")
print(f"class names also in the vocabulary: {int(class_names().word.isin(words().word).sum())}")
per_class = manifest(CLASS_IMAGES).groupby("wnid").size()
print(f"class images: {per_class.sum():,} in {len(per_class):,} classes, at least {per_class.min()} per class")

## Backgrounds that show the named class

Matched cosine similarity over the class names that a background shows, with word directions built from all backgrounds and without the background that shows the named class.

In [ ]:
shown = sweep_rows(checks).assign(change=lambda f: f.with_all_slots - f.without_own_slot)
largest = shown.loc[shown.change.abs().idxmax()]
backgrounds = int(shown.max_backgrounds_per_word.max())
print(f"class names shown by a background: {largest.n_words}, each on {backgrounds} background")
print(f"backgrounds the rebuilt word directions use: {n_slots() - backgrounds}")
print(f"largest absolute change: {abs(largest.change):.6f}, in {largest.label}")
(shown.set_index("label")[["with_all_slots", "without_own_slot", "change"]]
 .rename(columns={"with_all_slots": "all backgrounds", "without_own_slot": "without the matching background"})
 .rename_axis("Model"))

## Class diagnostics in DINOv3-H

Within-class cosine similarity and the cosine similarity with the nearest other visual direction, over the named classes.

In [ ]:
named = classes[classes.model.eq("dinov3-h") & classes.word.ne("")].set_index("word")
named = named.assign(nearest_class=named.nearest_wnid.map(imagenet.class_names()))
print(f"median within-class cosine similarity over the {len(named)} named classes: {named.coherence.median():.2f}")
lowest = named.coherence.rank()
for word in ("bee", "church"):
    print(f"within-class cosine similarity, {word}: {named.coherence[word]:.3f}, "
          f"rank {lowest[word]:.0f} from the lowest")
highest = named.nearest_cosine.rank(ascending=False)
for word in ("corn", "laptop"):
    print(f"nearest other visual direction to {word}: class '{named.nearest_class[word]}', "
          f"{named.nearest_word[word] or 'no selected name'}, cosine similarity "
          f"{named.nearest_cosine[word]:.3f}, rank {highest[word]:.0f} from the highest")